<a href="https://colab.research.google.com/github/yosungcho/yosungcho.github.io/blob/main/090826RetinalAgeHeadTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
"""
============================================================================
Retinal Age Head Training on BRSET — Google Colab Script
============================================================================

Purpose:
    Train an EfficientNet-B2 regression model to predict chronological age
    from fundus photographs (BRSET dataset). The trained model becomes the
    "retinal age head" of the FundusView multi-head oculomic platform, and
    the retinal age gap (predicted - actual) serves as a vascular-based
    cognitive risk warning indicator.

Assumed setup:
    - This script lives at:
        /content/drive/MyDrive/FundusView/retinal_age/retinal_age_training_colab.py
    - BRSET data lives at:
        /content/drive/MyDrive/FundusView/BRSET/
      containing:
        - fundus_photos/  (16,266 JPEG images)
        - labels.csv      (metadata with patient_age column)
    - Colab notebook mounts Drive, then does:
        exec(open('/content/drive/MyDrive/FundusView/retinal_age/retinal_age_training_colab.py').read())
      OR loads this file as a notebook cell.

Author: Yosung
Project: FundusView / Jojo CNN oculomic platform extension
============================================================================
"""

# ============================================================================
# SECTION 1: SETUP AND CONFIGURATION
# ============================================================================

import os
import sys
import glob
import json
import time
from datetime import datetime

# ---------- Path configuration (edit if your Drive structure differs) ----------
DRIVE_ROOT = '/content/drive/MyDrive/FundusView'
BRSET_DIR = os.path.join(DRIVE_ROOT, 'BRSET')
IMAGE_DIR = os.path.join(BRSET_DIR, 'fundus_photos')
LABELS_PATH = os.path.join(BRSET_DIR, 'labels.csv')

PROJECT_DIR = os.path.join(DRIVE_ROOT, 'retinal_age')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
LOG_DIR = os.path.join(PROJECT_DIR, 'logs')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# ---------- Training configuration ----------
CONFIG = {
    'backbone': 'efficientnet_b2',        # matches Cheung 2022 paper
    'image_size': 512,                     # standard for fundus AI
    'batch_size': 32,                      # good for T4 (16GB); drop to 16 if OOM
    'num_epochs': 30,
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'num_workers': 2,                      # Colab often works better with 2 than 4
    'seed': 42,
    'val_split': 0.15,
    'test_split': 0.15,
    'save_every_epoch': True,              # save checkpoint every epoch to survive disconnects
    'use_pretrained': True,
    'dropout': 0.3,
    'early_stopping_patience': 8,          # stop if val doesn't improve for N epochs
}

# ---------- Timestamp for this run ----------
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_LOG = os.path.join(LOG_DIR, f'run_{RUN_ID}.log')

def log(msg):
    """Print to stdout and append to log file (both survive disconnects if log is on Drive)."""
    timestamp = datetime.now().strftime('%H:%M:%S')
    line = f"[{timestamp}] {msg}"
    print(line)
    with open(RUN_LOG, 'a') as f:
        f.write(line + '\n')

log(f"=== Retinal age training run {RUN_ID} started ===")
log(f"Config: {json.dumps(CONFIG, indent=2)}")


# ============================================================================
# SECTION 2: DEPENDENCIES
# ============================================================================

# Colab has PyTorch pre-installed; we just need a couple extras
os.system('pip install timm grad-cam --quiet')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
from sklearn.model_selection import GroupShuffleSplit
from tqdm import tqdm
import matplotlib.pyplot as plt
import timm

# Reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    log(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    log("WARNING: No GPU detected. Training will be very slow. Enable GPU: Runtime > Change runtime type > T4 GPU")


# ============================================================================
# SECTION 3: DATA PREPARATION
# ============================================================================

log("Loading BRSET labels...")
df = pd.read_csv(LABELS_PATH)
log(f"Total rows in labels.csv: {len(df)}")
log(f"Columns: {list(df.columns)}")

# BRSET's age column is 'patient_age'. If the column name differs in your version,
# update this. Also update image filename column if it's not 'image_id'.
AGE_COLUMN = 'patient_age'
IMAGE_COLUMN = 'image_id'
PATIENT_COLUMN = 'patient_id'

# Inspect columns and fail loudly if the expected ones aren't there
for col in [AGE_COLUMN, IMAGE_COLUMN, PATIENT_COLUMN]:
    if col not in df.columns:
        log(f"ERROR: expected column '{col}' not found in labels.csv")
        log(f"Available columns: {list(df.columns)}")
        raise KeyError(f"Column '{col}' not in labels")

# Clean data: drop rows without age
before = len(df)
df = df.dropna(subset=[AGE_COLUMN]).copy()
df[AGE_COLUMN] = df[AGE_COLUMN].astype(float)
log(f"Dropped {before - len(df)} rows with missing age; {len(df)} remain")

# Sanity check age distribution
log(f"Age stats: min={df[AGE_COLUMN].min():.0f}, max={df[AGE_COLUMN].max():.0f}, "
    f"mean={df[AGE_COLUMN].mean():.1f}, median={df[AGE_COLUMN].median():.1f}")

# Verify image files actually exist for a sample (catches path issues early)
sample_check = df.sample(min(10, len(df)), random_state=42)
missing = 0
for _, row in sample_check.iterrows():
    img_path = os.path.join(IMAGE_DIR, row[IMAGE_COLUMN])
    # BRSET images may lack extension in labels; try both
    if not os.path.exists(img_path):
        if not any(os.path.exists(img_path + ext) for ext in ['.jpg', '.jpeg', '.png']):
            missing += 1
if missing > 0:
    log(f"WARNING: {missing}/10 sampled image files not found. Check IMAGE_DIR and IMAGE_COLUMN.")
    log(f"Example expected path: {os.path.join(IMAGE_DIR, sample_check.iloc[0][IMAGE_COLUMN])}")

# Patient-based split (CRITICAL: prevents data leakage from same patient in train and val)
log("Splitting data by patient_id (not image_id) to prevent leakage...")

gss_test = GroupShuffleSplit(n_splits=1, test_size=CONFIG['test_split'], random_state=CONFIG['seed'])
train_val_idx, test_idx = next(gss_test.split(df, groups=df[PATIENT_COLUMN]))
df_trainval = df.iloc[train_val_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

# Split trainval into train and val (val_split is fraction of trainval, not of total)
val_frac_of_trainval = CONFIG['val_split'] / (1 - CONFIG['test_split'])
gss_val = GroupShuffleSplit(n_splits=1, test_size=val_frac_of_trainval, random_state=CONFIG['seed'])
train_idx, val_idx = next(gss_val.split(df_trainval, groups=df_trainval[PATIENT_COLUMN]))
df_train = df_trainval.iloc[train_idx].reset_index(drop=True)
df_val = df_trainval.iloc[val_idx].reset_index(drop=True)

log(f"Train: {len(df_train)} images from {df_train[PATIENT_COLUMN].nunique()} patients")
log(f"Val:   {len(df_val)} images from {df_val[PATIENT_COLUMN].nunique()} patients")
log(f"Test:  {len(df_test)} images from {df_test[PATIENT_COLUMN].nunique()} patients")

# Verify no patient overlap
train_patients = set(df_train[PATIENT_COLUMN])
val_patients = set(df_val[PATIENT_COLUMN])
test_patients = set(df_test[PATIENT_COLUMN])
assert len(train_patients & val_patients) == 0, "PATIENT LEAKAGE: train/val overlap!"
assert len(train_patients & test_patients) == 0, "PATIENT LEAKAGE: train/test overlap!"
assert len(val_patients & test_patients) == 0, "PATIENT LEAKAGE: val/test overlap!"
log("Patient split verified: no leakage across train/val/test")

# Save the splits so results are reproducible
df_train.to_csv(os.path.join(RESULTS_DIR, f'split_train_{RUN_ID}.csv'), index=False)
df_val.to_csv(os.path.join(RESULTS_DIR, f'split_val_{RUN_ID}.csv'), index=False)
df_test.to_csv(os.path.join(RESULTS_DIR, f'split_test_{RUN_ID}.csv'), index=False)


# ============================================================================
# SECTION 4: DATASET AND TRANSFORMS
# ============================================================================

# ImageNet normalization (matches pretrained EfficientNet expectations)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class BRSETRetinalAgeDataset(Dataset):
    """
    Yields (image_tensor, age_float) pairs from BRSET.
    Handles both cases where image_id includes extension and where it doesn't.
    """
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row[IMAGE_COLUMN]

        # Try direct path first, then common extensions
        image_path = os.path.join(self.image_dir, str(image_id))
        if not os.path.exists(image_path):
            for ext in ['.jpg', '.jpeg', '.png']:
                candidate = image_path + ext
                if os.path.exists(candidate):
                    image_path = candidate
                    break
            else:
                raise FileNotFoundError(f"Image not found for id={image_id} in {self.image_dir}")

        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        age = torch.tensor(row[AGE_COLUMN], dtype=torch.float32)
        return image, age


train_dataset = BRSETRetinalAgeDataset(df_train, IMAGE_DIR, transform=train_transform)
val_dataset = BRSETRetinalAgeDataset(df_val, IMAGE_DIR, transform=eval_transform)
test_dataset = BRSETRetinalAgeDataset(df_test, IMAGE_DIR, transform=eval_transform)

# Sanity check: load one sample and verify
log("Loading one sample to verify dataset works...")
sample_img, sample_age = train_dataset[0]
log(f"Sample image tensor shape: {sample_img.shape}, age: {sample_age.item()}")

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
                        shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'],
                         shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)


# ============================================================================
# SECTION 5: MODEL
# ============================================================================

class RetinalAgeModel(nn.Module):
    """
    EfficientNet-B2 backbone with a regression head predicting age in years.
    This is designed to slot into the FundusView multi-head architecture as
    an additional head alongside DR / Glaucoma / Cataract / HR / AMD / RVO.
    """
    def __init__(self, backbone_name='efficientnet_b2', pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            num_classes=0,          # remove default classifier
            global_pool='avg'
        )
        backbone_dim = self.backbone.num_features

        self.age_head = nn.Sequential(
            nn.Linear(backbone_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        features = self.backbone(x)
        age = self.age_head(features)
        return age.squeeze(-1)


model = RetinalAgeModel(
    backbone_name=CONFIG['backbone'],
    pretrained=CONFIG['use_pretrained'],
    dropout=CONFIG['dropout']
).to(device)

n_params = sum(p.numel() for p in model.parameters())
log(f"Model: {CONFIG['backbone']}, {n_params:,} parameters")


# ============================================================================
# SECTION 6: TRAINING SETUP
# ============================================================================

optimizer = AdamW(model.parameters(),
                  lr=CONFIG['learning_rate'],
                  weight_decay=CONFIG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['num_epochs'])

# L1Loss = MAE, which is directly interpretable (average years off) and
# more robust to outliers than MSE for age prediction
criterion = nn.L1Loss()


# ============================================================================
# SECTION 7: RESUME FROM CHECKPOINT (survives Colab disconnects)
# ============================================================================

def find_latest_checkpoint():
    """Find the most recent epoch checkpoint from any prior run."""
    pattern = os.path.join(CHECKPOINT_DIR, 'checkpoint_epoch_*.pth')
    checkpoints = sorted(glob.glob(pattern))
    return checkpoints[-1] if checkpoints else None


start_epoch = 0
best_val_mae = float('inf')
epochs_without_improvement = 0
history = {'train_mae': [], 'val_mae': [], 'lr': []}

latest_ckpt = find_latest_checkpoint()
if latest_ckpt:
    log(f"Found existing checkpoint: {latest_ckpt}")
    log("Resuming training. To start fresh, delete files in CHECKPOINT_DIR.")
    ckpt = torch.load(latest_ckpt, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_mae = ckpt['best_val_mae']
    epochs_without_improvement = ckpt.get('epochs_without_improvement', 0)
    history = ckpt.get('history', history)
    log(f"Resuming from epoch {start_epoch}, best val MAE so far: {best_val_mae:.3f}")
else:
    log("No checkpoint found; starting fresh training")


# ============================================================================
# SECTION 8: TRAINING LOOP
# ============================================================================

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    losses = []
    pbar = tqdm(loader, desc="Train", leave=False)
    for images, ages in pbar:
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)

        optimizer.zero_grad()
        preds = model(images)
        loss = criterion(preds, ages)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        pbar.set_postfix({'MAE': f'{loss.item():.2f}'})
    return sum(losses) / len(losses)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    losses = []
    all_preds, all_actuals = [], []
    for images, ages in tqdm(loader, desc="Eval", leave=False):
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)
        preds = model(images)
        loss = criterion(preds, ages)
        losses.append(loss.item())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_actuals.extend(ages.cpu().numpy().tolist())
    mae = sum(losses) / len(losses)
    return mae, np.array(all_preds), np.array(all_actuals)


log(f"=== Starting training from epoch {start_epoch} to {CONFIG['num_epochs']} ===")
training_start_time = time.time()

for epoch in range(start_epoch, CONFIG['num_epochs']):
    epoch_start = time.time()

    train_mae = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_mae, _, _ = evaluate(model, val_loader, criterion, device)

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    history['train_mae'].append(train_mae)
    history['val_mae'].append(val_mae)
    history['lr'].append(current_lr)

    epoch_time = time.time() - epoch_start
    log(f"Epoch {epoch+1}/{CONFIG['num_epochs']} | "
        f"Train MAE: {train_mae:.3f}y | Val MAE: {val_mae:.3f}y | "
        f"LR: {current_lr:.2e} | Time: {epoch_time:.0f}s")

    # Save checkpoint every epoch (survives disconnect)
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch:03d}.pth')
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_mae': best_val_mae,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history,
        'config': CONFIG,
    }, ckpt_path)

    # Save best model separately (this is the one to use for inference)
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        epochs_without_improvement = 0
        best_path = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_mae': val_mae,
            'config': CONFIG,
        }, best_path)
        log(f"  → New best val MAE {val_mae:.3f}y saved to best_model.pth")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CONFIG['early_stopping_patience']:
            log(f"Early stopping: no improvement for {CONFIG['early_stopping_patience']} epochs")
            break

    # Housekeeping: keep only last 3 epoch checkpoints + best_model to save Drive space
    all_epoch_ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'checkpoint_epoch_*.pth')))
    for old_ckpt in all_epoch_ckpts[:-3]:
        os.remove(old_ckpt)

total_time = time.time() - training_start_time
log(f"=== Training complete in {total_time/60:.1f} minutes ===")
log(f"Best validation MAE: {best_val_mae:.3f} years")


# ============================================================================
# SECTION 9: TEST SET EVALUATION AND RETINAL AGE GAP ANALYSIS
# ============================================================================

log("=== Loading best model for test evaluation ===")
best_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pth'), map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])
log(f"Loaded best model from epoch {best_ckpt['epoch']} with val MAE {best_ckpt['val_mae']:.3f}y")

test_mae, test_preds, test_actuals = evaluate(model, test_loader, criterion, device)
log(f"Test set MAE: {test_mae:.3f} years")

# Retinal age gap: predicted - actual
gaps = test_preds - test_actuals

log(f"=== Retinal age gap distribution on test set ===")
log(f"  Mean gap: {gaps.mean():.2f} years (ideally near 0)")
log(f"  Std gap:  {gaps.std():.2f} years")
log(f"  5th percentile:  {np.percentile(gaps, 5):.2f}")
log(f"  25th percentile: {np.percentile(gaps, 25):.2f}")
log(f"  50th percentile: {np.percentile(gaps, 50):.2f}")
log(f"  75th percentile: {np.percentile(gaps, 75):.2f}")
log(f"  95th percentile: {np.percentile(gaps, 95):.2f}")

# What fraction of test set falls in each proposed warning tier?
# These are the tentative thresholds — real calibration comes later
tier_normal = (gaps < 3).sum() / len(gaps) * 100
tier_uncertain = ((gaps >= 3) & (gaps < 8)).sum() / len(gaps) * 100
tier_alert = (gaps >= 8).sum() / len(gaps) * 100
log(f"=== Tentative warning tier distribution ===")
log(f"  Normal (gap < 3y):     {tier_normal:.1f}%")
log(f"  Uncertain (3-8y):      {tier_uncertain:.1f}%")
log(f"  Alert (gap >= 8y):     {tier_alert:.1f}%")

# Save test predictions for later analysis
test_results_df = df_test.copy()
test_results_df['predicted_age'] = test_preds
test_results_df['retinal_age_gap'] = gaps
test_results_df.to_csv(os.path.join(RESULTS_DIR, f'test_predictions_{RUN_ID}.csv'), index=False)
log(f"Test predictions saved to test_predictions_{RUN_ID}.csv")


# ============================================================================
# SECTION 10: DIAGNOSTIC PLOTS
# ============================================================================

log("=== Generating diagnostic plots ===")

# Plot 1: Training curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, len(history['train_mae']) + 1)
ax[0].plot(epochs_range, history['train_mae'], label='Train MAE', marker='o')
ax[0].plot(epochs_range, history['val_mae'], label='Val MAE', marker='s')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('MAE (years)')
ax[0].set_title('Training and Validation MAE')
ax[0].legend()
ax[0].grid(True, alpha=0.3)

ax[1].plot(epochs_range, history['lr'], marker='.')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Learning Rate')
ax[1].set_title('Learning Rate Schedule')
ax[1].set_yscale('log')
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'training_curves_{RUN_ID}.png'), dpi=150)
plt.close()

# Plot 2: Predicted vs actual age (scatter)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(test_actuals, test_preds, alpha=0.3, s=10)
min_age = min(test_actuals.min(), test_preds.min())
max_age = max(test_actuals.max(), test_preds.max())
ax[0].plot([min_age, max_age], [min_age, max_age], 'r--', label='Perfect prediction')
ax[0].set_xlabel('Actual Age (years)')
ax[0].set_ylabel('Predicted Retinal Age (years)')
ax[0].set_title(f'Test Set: Predicted vs Actual (MAE = {test_mae:.2f}y)')
ax[0].legend()
ax[0].grid(True, alpha=0.3)

# Plot 3: Retinal age gap histogram
ax[1].hist(gaps, bins=50, edgecolor='black', alpha=0.7)
ax[1].axvline(0, color='green', linestyle='--', label='No gap')
ax[1].axvline(3, color='orange', linestyle='--', label='Uncertain threshold (3y)')
ax[1].axvline(8, color='red', linestyle='--', label='Alert threshold (8y)')
ax[1].set_xlabel('Retinal Age Gap (predicted - actual, years)')
ax[1].set_ylabel('Count')
ax[1].set_title('Retinal Age Gap Distribution on Test Set')
ax[1].legend()
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'test_analysis_{RUN_ID}.png'), dpi=150)
plt.close()

log("Diagnostic plots saved to RESULTS_DIR")


# ============================================================================
# SECTION 11: GRAD-CAM SANITY CHECK
# ============================================================================

log("=== Running Grad-CAM sanity check ===")
log("This verifies the model is looking at vessels, not artifacts")

try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image

    # For EfficientNet, target the last conv block
    target_layer = model.backbone.conv_head if hasattr(model.backbone, 'conv_head') else model.backbone.blocks[-1]

    cam = GradCAM(model=model, target_layers=[target_layer])

    n_samples = 6
    sample_indices = np.linspace(0, len(test_dataset) - 1, n_samples, dtype=int)

    fig, axes = plt.subplots(n_samples, 2, figsize=(10, 4 * n_samples))
    for i, idx in enumerate(sample_indices):
        image, actual_age = test_dataset[idx]
        input_tensor = image.unsqueeze(0).to(device)

        # Predicted age for this specific image
        with torch.no_grad():
            pred_age = model(input_tensor).item()

        grayscale_cam = cam(input_tensor=input_tensor)[0]

        # Denormalize image for display
        display = image.permute(1, 2, 0).cpu().numpy()
        display = (display * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)

        visualization = show_cam_on_image(display, grayscale_cam, use_rgb=True)

        axes[i, 0].imshow(display)
        axes[i, 0].set_title(f"Actual: {actual_age.item():.0f}y | Predicted: {pred_age:.0f}y | Gap: {pred_age - actual_age.item():+.1f}y")
        axes[i, 0].axis('off')
        axes[i, 1].imshow(visualization)
        axes[i, 1].set_title("Grad-CAM attention (should be on vessels)")
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'gradcam_sanity_{RUN_ID}.png'), dpi=150)
    plt.close()
    log("Grad-CAM plots saved. Manually inspect: attention should be on vessels, optic disc, macula")
    log("Red flags: attention on borders, black background, or lens artifacts")

except Exception as e:
    log(f"Grad-CAM failed (non-fatal): {e}")
    log("Training results are still valid; only the interpretability plot was skipped")


# ============================================================================
# SECTION 12: FINAL SUMMARY
# ============================================================================

summary = {
    'run_id': RUN_ID,
    'config': CONFIG,
    'best_val_mae_years': float(best_val_mae),
    'test_mae_years': float(test_mae),
    'test_gap_mean': float(gaps.mean()),
    'test_gap_std': float(gaps.std()),
    'n_train': len(df_train),
    'n_val': len(df_val),
    'n_test': len(df_test),
    'n_epochs_completed': len(history['train_mae']),
    'total_training_minutes': total_time / 60,
    'tier_percentages': {
        'normal_lt_3y': float(tier_normal),
        'uncertain_3_to_8y': float(tier_uncertain),
        'alert_gte_8y': float(tier_alert),
    },
    'files': {
        'best_model': os.path.join(CHECKPOINT_DIR, 'best_model.pth'),
        'test_predictions': os.path.join(RESULTS_DIR, f'test_predictions_{RUN_ID}.csv'),
        'training_curves': os.path.join(RESULTS_DIR, f'training_curves_{RUN_ID}.png'),
        'test_analysis': os.path.join(RESULTS_DIR, f'test_analysis_{RUN_ID}.png'),
        'gradcam': os.path.join(RESULTS_DIR, f'gradcam_sanity_{RUN_ID}.png'),
        'log': RUN_LOG,
    }
}

summary_path = os.path.join(RESULTS_DIR, f'summary_{RUN_ID}.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

log(f"=== RUN COMPLETE ===")
log(f"Summary saved to: {summary_path}")
log(f"Best model at:    {os.path.join(CHECKPOINT_DIR, 'best_model.pth')}")
log(f"")
log(f"Next steps:")
log(f"  1. Inspect Grad-CAM plot: attention should land on vessels")
log(f"  2. Check test_analysis.png: scatter should hug diagonal")
log(f"  3. If val MAE > 6y, consider: more epochs, larger image size, or lower learning rate")
log(f"  4. If val MAE < 4y, proceed to integration into FundusView multi-head architecture")
log(f"  5. Add AutoMorph feature regression heads (vessel density, fractal dim, AVR) for stronger biomarker set")

[15:41:24] === Retinal age training run 20260908_154124 started ===
[15:41:24] Config: {
  "backbone": "efficientnet_b2",
  "image_size": 512,
  "batch_size": 32,
  "num_epochs": 30,
  "learning_rate": 0.0001,
  "weight_decay": 0.0001,
  "num_workers": 2,
  "seed": 42,
  "val_split": 0.15,
  "test_split": 0.15,
  "save_every_epoch": true,
  "use_pretrained": true,
  "dropout": 0.3,
  "early_stopping_patience": 8
}
[15:41:26] GPU: Tesla T4, VRAM: 15.6 GB
[15:41:26] Loading BRSET labels...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/FundusView/BRSET/labels.csv'